# Run analysis

Reads the CSVs a run writes into its own directory:

- `client_metrics_global.csv` - the client-side ledger, flushed every second. Rows are **cumulative** per (node, chosen SLA, executed level) cell; every plot below diffs consecutive flushes per cell to get per-interval rates. All cross-arm comparison numbers come from here.
- `occupancy_<n>.csv` - per node, per 100 ms control interval: utilization U (slot-time / S_max), average in-flight, and the shadow price lambda.
- `histograms_<n>.csv` - per node, every 5 s: the server-side service-time histogram cells per (level, gap bucket).
- `config.json` - the exact config of this run (mode, phases), archived by run_all.sh.

Run all cells top to bottom; the setup cell loads everything the others use.

In [ ]:
# Setup: load the run's data
import glob, json, os
import pandas as pd
import matplotlib.pyplot as plt

CSV_DIR = "."

with open(os.path.join(CSV_DIR, "config.json")) as f:
    CONFIG = json.load(f)
MODE = CONFIG.get("mode", "?")
PHASES = CONFIG.get("phases", [])
SINGLE_PHASE = CONFIG.get("experiment", {}).get("runSinglePhase", False)

ledger = pd.read_csv(os.path.join(CSV_DIR, "client_metrics_global.csv"))
T0 = ledger["Timestamp"].min()
ledger["Time_s"] = (ledger["Timestamp"] - T0) / 1000.0

occupancy = {}
for path in sorted(glob.glob(os.path.join(CSV_DIR, "occupancy_*.csv"))):
    node = int(path.split("_")[-1].split(".")[0])
    df = pd.read_csv(path)
    df["Time_s"] = (df["Timestamp"] - T0) / 1000.0
    occupancy[node] = df

histograms = {}
for path in sorted(glob.glob(os.path.join(CSV_DIR, "histograms_*.csv"))):
    node = int(path.split("_")[-1].split(".")[0])
    df = pd.read_csv(path)
    df["Time_s"] = (df["Timestamp"] - T0) / 1000.0
    histograms[node] = df

CELL_KEY = ["NodeId", "ChosenLevel", "ExecutedLevel"]

def interval_deltas(columns):
    """Ledger rows are cumulative per cell: diff consecutive flushes per
    cell, then sum per flush timestamp for system-wide per-interval deltas."""
    df = ledger.sort_values("Timestamp").copy()
    for c in columns:
        df[c] = df.groupby(CELL_KEY)[c].diff().fillna(df[c])
    out = df.groupby("Timestamp")[columns].sum().reset_index()
    out["Time_s"] = (out["Timestamp"] - T0) / 1000.0
    out["dt_s"] = out["Timestamp"].diff().fillna(1000) / 1000.0
    return out

def phase_lines(ax):
    if SINGLE_PHASE or not PHASES:
        return
    t = 0
    for p in PHASES[:-1]:
        t += p["durationSeconds"]
        ax.axvline(t, color="gray", linestyle=":", linewidth=0.8)

print(f"mode={MODE}  nodes={len(occupancy)}  ledger rows={len(ledger)}  "
      f"phases={[p['name'] for p in PHASES]}")

In [ ]:
# Throughput: served and rejected per second
rates = interval_deltas(["CountTotal", "RejectedTotal", "LostTotal", "FallbacksTotal"])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rates["Time_s"], rates["CountTotal"] / rates["dt_s"], label="served/s", color="tab:blue")
ax.plot(rates["Time_s"], rates["RejectedTotal"] / rates["dt_s"], label="rejected/s", color="tab:red")
ax.plot(rates["Time_s"], rates["FallbacksTotal"] / rates["dt_s"], label="fallbacks/s", color="tab:orange")
phase_lines(ax)
ax.set_xlabel("time [s]"); ax.set_ylabel("requests/s")
ax.set_title(f"Throughput and shedding ({MODE})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Profit: predicted vs realized per second
profit = interval_deltas(["PredictedProfitSum", "RealizedProfitSum"])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(profit["Time_s"], profit["PredictedProfitSum"] / profit["dt_s"], label="predicted/s", color="tab:gray")
ax.plot(profit["Time_s"], profit["RealizedProfitSum"] / profit["dt_s"], label="realized/s", color="tab:green")
phase_lines(ax)
ax.set_xlabel("time [s]"); ax.set_ylabel("profit/s")
ax.set_title(f"Predicted vs realized profit ({MODE}) - the gap is the misprediction stream")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Capacity: utilization and the shadow price per node
# This replaces the old token-bucket plots: capacity is slot-time now.
fig, (ax_u, ax_l) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for node, df in occupancy.items():
    ax_u.plot(df["Time_s"], df["U"].rolling(10, min_periods=1).mean(), label=f"node {node}", linewidth=0.9)
    ax_l.plot(df["Time_s"], df["Lambda"], label=f"node {node}", linewidth=0.9)
ax_u.axhline(CONFIG["chameleon"]["uTarget"], color="black", linestyle="--", linewidth=0.8, label="u_target")
ax_u.set_ylabel("utilization U"); ax_u.legend(ncol=3); ax_u.grid(alpha=0.3)
ax_l.set_yscale("log"); ax_l.set_ylabel("lambda (log)"); ax_l.set_xlabel("time [s]"); ax_l.grid(alpha=0.3)
phase_lines(ax_u); phase_lines(ax_l)
ax_u.set_title(f"Occupancy utilization (1 s smoothed) and shadow price ({MODE})")
plt.tight_layout(); plt.show()

In [ ]:
# Consistency mix: executed level per second
df = ledger.sort_values("Timestamp").copy()
df["Count_d"] = df.groupby(CELL_KEY)["CountTotal"].diff().fillna(df["CountTotal"])
mix = df[df["ExecutedLevel"] != "-"].groupby(["Timestamp", "ExecutedLevel"])["Count_d"].sum().unstack(fill_value=0)
mix.index = (mix.index - T0) / 1000.0
dt = pd.Series(mix.index).diff().fillna(1.0).values
fig, ax = plt.subplots(figsize=(12, 4))
for level in sorted(mix.columns):
    ax.plot(mix.index, mix[level] / dt, label=level, linewidth=0.9)
phase_lines(ax)
ax.set_xlabel("time [s]"); ax.set_ylabel("requests/s")
ax.set_title(f"Executed level mix ({MODE}) - the mix is emergent, nothing configures it")
ax.legend(ncol=3, fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Client-observed latency: interval average + final tails per busy cell
lat = ledger.sort_values("Timestamp").copy()
lat["LatencySum"] = lat["AvgLatencyMs"] * lat["CountTotal"]
lat["LatencySum_d"] = lat.groupby(CELL_KEY)["LatencySum"].diff().fillna(lat["LatencySum"])
lat["Count_d"] = lat.groupby(CELL_KEY)["CountTotal"].diff().fillna(lat["CountTotal"])
by_t = lat.groupby("Timestamp")[["LatencySum_d", "Count_d"]].sum()
by_t["avg_ms"] = by_t["LatencySum_d"] / by_t["Count_d"].clip(lower=1)
by_t.index = (by_t.index - T0) / 1000.0
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(by_t.index, by_t["avg_ms"], color="tab:purple")
phase_lines(ax)
ax.set_xlabel("time [s]"); ax.set_ylabel("avg latency [ms]")
ax.set_title(f"Client-observed average latency per interval ({MODE})")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Cumulative tail percentiles from the final flush (per cell, count-ranked).
final = ledger[ledger["Timestamp"] == ledger["Timestamp"].max()]
busy = final[final["CountTotal"] > 0.01 * final["CountTotal"].sum()]
cols = ["NodeId", "ChosenLevel", "ExecutedLevel", "CountTotal", "AvgLatencyMs", "P50Ms", "P95Ms", "P99Ms"]
print(busy.sort_values("CountTotal", ascending=False)[cols].to_string(index=False))

In [ ]:
# Upgrades: free vs waiting fraction, and the satisfied-rung distribution
up = interval_deltas(["UpgradesFreeTotal", "UpgradesWaitingTotal"])
total_up = (up["UpgradesFreeTotal"] + up["UpgradesWaitingTotal"]).clip(lower=1)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(up["Time_s"], up["UpgradesFreeTotal"] / total_up, label="free fraction", color="tab:green")
ax.plot(up["Time_s"], up["UpgradesWaitingTotal"] / total_up, label="waiting fraction", color="tab:orange")
phase_lines(ax)
ax.set_xlabel("time [s]"); ax.set_ylabel("fraction of upgrades"); ax.set_ylim(0, 1)
ax.set_title(f"Free vs waiting upgrades ({MODE})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Satisfied rung per (chosen SLA): the head-to-head metric against Pileus.
final = ledger[ledger["Timestamp"] == ledger["Timestamp"].max()]
rungs = final.groupby("ChosenLevel")[["SatisfiedRung0", "SatisfiedRung1", "SatisfiedRung2",
                                      "SatisfiedRung3", "SatisfiedNone"]].sum()
rungs = rungs[rungs.sum(axis=1) > 0]
print(rungs.to_string())

In [ ]:
# Summary report (copy-paste friendly)
final = ledger[ledger["Timestamp"] == ledger["Timestamp"].max()]
served = final["CountTotal"].sum()
duration_s = (ledger["Timestamp"].max() - T0) / 1000.0
print("=" * 64)
print(f"RUN SUMMARY  mode={MODE}  duration={duration_s:.0f}s")
print("=" * 64)
print(f"served:            {served:,.0f}  ({served / max(duration_s, 1):,.0f}/s avg)")
print(f"rejected:          {final['RejectedTotal'].sum():,.0f}")
print(f"lost:              {final['LostTotal'].sum():,.0f}")
print(f"fallbacks:         {final['FallbacksTotal'].sum():,.0f}")
print(f"redirects:         {final['RedirectsTotal'].sum():,.0f}")
print(f"violations:        {final['SessionViolationsTotal'].sum():,.0f}")
print(f"predicted profit:  {final['PredictedProfitSum'].sum():,.0f}")
print(f"realized profit:   {final['RealizedProfitSum'].sum():,.0f}")
up_free = final["UpgradesFreeTotal"].sum(); up_wait = final["UpgradesWaitingTotal"].sum()
print(f"upgrades:          {up_free + up_wait:,.0f}  (free {up_free / max(up_free + up_wait, 1):.1%})")
for node, df in sorted(occupancy.items()):
    print(f"node {node}: meanU={df['U'].mean():.3f}  maxU={df['U'].max():.2f}  "
          f"maxLambda={df['Lambda'].max():.4f}")